# CS383: Data Science and Machine Learning
## Lecture 6 Exercises — Feature Engineering

Fill in every `__________` blank, then run all cells top to bottom. When you've completed this
notebook, download it (File → Save and Export Notebook As → Notebook (.ipynb), or the **Download**
button in the toolbar) and submit it on BrightSpace under **Lecture 6 Exercise** as a Jupyter Notebook
(.ipynb) file.

### Setup — NYC restaurant inspections

Run this first — it rebuilds the same restaurant inspections dataset from the lecture.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "grade"]).reset_index(drop=True)

    # One row here is one violation citation, not one full inspection -- a single inspection can
    # contribute more than one row.
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 1200
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    is_critical = rng.integers(0, 2, size=n)
    grade = rng.choice(["A", "B", "C"], size=n, p=[0.6, 0.25, 0.15])
    score = rng.integers(0, 71, size=n)

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "score": score,
        "grade": grade,
        "critical_flag": np.where(is_critical == 1, "Critical", "Not Critical"),
        "is_critical": is_critical,
    })
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df.head()

---

## Exercise 1 — Prep a Feature Set the Right Way

**Scenario:** you're preparing `boro` (nominal), `grade` (ordinal), and `is_critical` (numeric) as
model-ready features — the same kind of prep step your capstone will need. Follow the steps below in
the correct order to avoid leaking test data into your preprocessing.

### Step 1 — Split first

Before touching any scaler or encoder, split the data.

In [ ]:
X = inspections_df[["boro", "grade", "is_critical"]]
y = inspections_df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=__________, random_state=383
)

print("Training rows:", len(X_train))
print("Test rows:    ", len(X_test))

### Step 2 — One-hot encode `boro`

`boro` is nominal — no natural order — so it gets one-hot encoded. Fit the encoder on the training
data only.

In [ ]:
boro_encoder = OneHotEncoder(handle_unknown="ignore")

boro_train_encoded = boro_encoder.__________(X_train[["boro"]])
boro_test_encoded = boro_encoder.__________(X_test[["boro"]])

print("Training shape:", boro_train_encoded.shape)
print("Test shape:    ", boro_test_encoded.shape)

### Step 3 — Ordinal encode `grade`

`grade` has a real order: A is better than B is better than C. Give the encoder that exact order.

In [ ]:
grade_encoder = OrdinalEncoder(categories=[["__________", "B", "C"]])

grade_train_encoded = grade_encoder.fit_transform(X_train[["grade"]])
grade_test_encoded = grade_encoder.transform(X_test[["grade"]])

print("A few encoded training values:", grade_train_encoded[:5].ravel())

### Step 4 — Scale the numeric features, without leaking

Fit the scaler once, on training data only, then reuse it for the test data.

In [ ]:
numeric_cols = ["is_critical"]

scaler = StandardScaler()
numeric_train_scaled = scaler.__________(X_train[numeric_cols])   # fit AND transform
numeric_test_scaled = scaler.__________(X_test[numeric_cols])     # transform ONLY

print("Training means after scaling (should be ~0):", numeric_train_scaled.mean(axis=0).round(3))

### Step 5 — Explain it back

In 2-3 sentences, explain to someone unfamiliar with this lecture why Steps 2-4 all fit their encoders
and scalers on `X_train` only, rather than on the full `X` before splitting.

**Your explanation:**

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. Why does `grade` get ordinal encoding while `boro` gets one-hot encoding? What would go wrong if you swapped them?
2. In your own words, what is data leakage — and describe one concrete way it could happen with a scaler.
3. Why is "fit only on training data" the rule, rather than "fit on the whole dataset, then split"?
4. What problem does grouping rare categories into `"Other"` solve? What could go wrong if `MIN_COUNT` were set way too high?
5. What question do you still have about feature engineering heading into Week 7 (Regression)?

**Your responses:**

1.
2.
3.
4.
5. 

## Optional Challenge

Apply the same encode-and-scale-without-leaking workflow from Exercise 1 to a different dataset: NYC 311.

### Setup — NYC 311

In [ ]:
import os

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng.integers(0, n_days, size=n)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")
    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(complaint_types, size=n),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df[["complaint_type", "borough", "hour_filed", "resolution_time_hours"]].head()

### Step 1 — Split the data

Set aside `resolution_time_hours` as `y` (it'll be a real regression target in Week 7). Use
`complaint_type`, `borough`, and `hour_filed` as `X`.

In [ ]:
# Your code here


### Step 2 — Encode the categoricals

One-hot encode `complaint_type` and `borough`, fit on training data only.

In [ ]:
# Your code here


### Step 3 — Scale the numeric feature

Scale `hour_filed`, fit on training data only.

In [ ]:
# Your code here


### Step 4 — Reflect

Think ahead to your capstone project. What's one specific place in *your* dataset where leakage could
sneak in if you weren't careful — and what would you do to prevent it?

**Your answer:**

### Big idea
> Every model built from Week 7 onward is only as trustworthy as the preprocessing that fed it. "Fit on
> train, transform everywhere" is the one habit that prevents almost every leakage mistake — make it
> automatic now, before it's buried inside a bigger pipeline.